In [1]:
# Import all functions from the required modules
from cordo_sherpa_module import *
from cordo_ineris_module import *
from correction_module import *
from expo_functions_module import *
from morbidity_analysis_module import *
from plot_module import *

print("Successfully loaded all modules")

Successfully loaded plotting command
Successfully loaded all modules


In [2]:
#Load IRIS shp files
# Paths to the files
path_fichier_shp = "data/2-output-data/donnees_shp"
title_shp = "donnees_insee_iris"
path_fichier_pourcents = "data/2-output-data"
title_pourcents = "pourcents"

# Load the concentration points
conc_points = coordo_sherpa(sc="s1", pol="ug_NO2", year=2019)

# Load the exported data
donnees_exportees = gpd.read_file(os.path.join(path_fichier_shp, f"{title_shp}.shp"))

# Transform the CRS of the exported data to match the concentration points
donnees_exportees_transformed = donnees_exportees.to_crs(epsg=conc_points.crs.to_epsg())

# Check if CRSs are the same
if conc_points.crs == donnees_exportees_transformed.crs:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are the same.")
else:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are different.")
grille_combinee = gpd.read_file(os.path.join(path_fichier_pourcents, f"{title_pourcents}.shp"))
grille_combinee = grille_combinee.to_crs(conc_points.crs)

Concentrations in 2019 and 2019 are calculated for the pollutant 'ug_no2' (s1).
CRS for conc_points_transformed and donnees_exportees_transformed are the same.


In [3]:
#Load INSEE mortality and population data
# Define paths for shapefiles
path_fichier_shp = "data/2-output-data/donnees_shp"
path_fichier_shp_1 = "data/2-output-data/donnees_shp_1"
path_fichier_shp_2 = "data/2-output-data/donnees_shp_2"
path_fichier_shp_3 = "data/2-output-data/donnees_shp_3"
path_fichier_pourcents = "data/2-output-data"

# Titles for INSEE Data
title_shp = "donnees_insee_iris"
title_shp_1 = "donnees_insee_iris_toutage_1"
title_shp_2 = "donnees_insee_iris_toutage_2"
title_shp_3 = "donnees_insee_iris_toutage_3"
title_pourcents = "pourcents"

# Read shapefiles into GeoDataFrames
donnees_shp_1 = gpd.read_file(os.path.join(path_fichier_shp_1, f"{title_shp_1}.shp"))
donnees_shp_2 = gpd.read_file(os.path.join(path_fichier_shp_2, f"{title_shp_2}.shp"))
donnees_shp_3 = gpd.read_file(os.path.join(path_fichier_shp_3, f"{title_shp_3}.shp"))

# Combine the three GeoDataFrames
donnees_merged = gpd.GeoDataFrame(pd.concat([donnees_shp_1, donnees_shp_2, donnees_shp_3], ignore_index=True))

In [6]:
#Distribute health data by age and IRIS to the main file (for Life Table Approach)
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# --- Load health data ---
health_data = pd.read_csv(r"C:\Users\aysharma\Eqis_codes\data\Morbidity data check\final_health_data.csv")
# Keep INSEE as 5-character commune codes (baseline)
health_data['insee'] = health_data['insee'].astype(str).str.upper().str.zfill(5)

# --- Preprocess population and mortality data ---
df = donnees_merged.copy()

# Use 5-character commune codes as baseline from donnees_merged
df['comcod'] = df['comcod'].astype(str).str.upper().str.zfill(5)

# --- Map arrondissements to their main communes at IRIS level (5-char codes) ---
arrondissement_mapping = {
    **{str(code).zfill(5): "75056" for code in range(75101, 75121)},  # Paris
    **{str(code).zfill(5): "13055" for code in range(13201, 13217)},  # Marseille
    **{str(code).zfill(5): "69123" for code in range(69381, 69390)},  # Lyon
}
df['comcod'] = df['comcod'].map(lambda x: arrondissement_mapping.get(x, x))

# --- Diagnostics: commune coverage baseline vs health data ---
pop_communes = df['comcod'].dropna().unique()
health_communes = health_data['insee'].dropna().unique()

print(f"Baseline unique communes in donnees_merged (5-char): {len(pop_communes)}")
print(f"Unique communes in health_data (5-char): {len(health_communes)}")

pop_only = set(pop_communes) - set(health_communes)
health_only = set(health_communes) - set(pop_communes)
print(f"Communes present only in population: {len(pop_only)}")
print(f"Communes present only in health: {len(health_only)}")

# --- Merge IRIS×age population with commune health totals ---
# Keep baseline comcod universe from donnees_merged (LEFT join)
merged = df.merge(
    health_data[['insee', 'disease', 'absolute_incidence', 'inc_100000']],
    left_on="comcod",
    right_on="insee",
    how="left"
)

print(f"Communes retained after merge (based on donnees_merged): {merged['comcod'].nunique()}")

# --- Define disease-specific age ranges ---
disease_age_groups = {
    "Hypertension": (18, 99),
    "Lung Cancer": (35, 99),
    "Asthma in children": (0, 17),
    "Asthma in adult": (18, 39),
    "COPD": (40, 99),
    "ALRI in children": (0, 12),
    "Stroke": (35, 99),
    "IHD events": (30, 99),
    "Diabetes T2": (45, 99),
}

# --- Allocate commune totals to IRIS×age ---
merged["absolute_incidence_iris"] = 0.0

for disease, (amin, amax) in disease_age_groups.items():
    mask = (merged["disease"] == disease) & (merged["age"].between(amin, amax))
    # population denominator per commune
    denom = merged.loc[mask].groupby(["comcod", "disease"])["pop2019"].transform("sum")
    share = merged.loc[mask, "pop2019"] / denom.replace(0, np.nan)
    merged.loc[mask, "absolute_incidence_iris"] = (
            share.fillna(0) * merged.loc[mask, "absolute_incidence"].fillna(0)
    )

# --- Verification ---
print("\nDisease-specific totals before and after distribution:")
print("-" * 80)
print(f"{'Disease':<20} {'Before':>15} {'After':>15} {'Difference':>15} {'Relative %':>15}")
print("-" * 80)

for disease in disease_age_groups.keys():
    before = health_data.loc[health_data["disease"] == disease, "absolute_incidence"].sum()
    after = merged.loc[merged["disease"] == disease, "absolute_incidence_iris"].sum()
    diff = after - before
    rel = (diff / before * 100) if before > 0 else 0
    print(f"{disease:<20} {before:>15.2f} {after:>15.2f} {diff:>15.2f} {rel:>15.2f}")

total_before = health_data["absolute_incidence"].sum()
total_after = merged["absolute_incidence_iris"].sum()
diff = total_after - total_before
rel = (diff / total_before * 100) if total_before > 0 else 0
print("-" * 80)
print(f"{'TOTAL':<20} {total_before:>15.2f} {total_after:>15.2f} {diff:>15.2f} {rel:>15.2f}")


Baseline unique communes in donnees_merged (5-char): 33787
Unique communes in health_data (5-char): 35228
Communes present only in population: 0
Communes present only in health: 1441
Communes retained after merge (based on donnees_merged): 33787

Disease-specific totals before and after distribution:
--------------------------------------------------------------------------------
Disease                       Before           After      Difference      Relative %
--------------------------------------------------------------------------------
Hypertension               712414.50       697848.75       -14565.75           -2.04
Lung Cancer                 39634.80        38760.66         -874.14           -2.21
Asthma in children         201989.25       198422.25        -3567.00           -1.77
Asthma in adult             92322.25        90681.50        -1640.75           -1.78
COPD                       194491.25       190100.75        -4390.50           -2.26
ALRI in children          

In [4]:
#Distribute health data by age to commune level (no IRIS distribution) and keep aggregated at comcod
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# --- Load health data ---
health_data = pd.read_csv(r"C:\Users\aysharma\Eqis_codes\data\Morbidity data check\final_health_data.csv")
# Keep INSEE as 5-character commune codes (baseline)
health_data['insee'] = health_data['insee'].astype(str).str.upper().str.zfill(5)

# --- Preprocess population and mortality data ---
df = donnees_merged.copy()

# Use 5-character commune codes as baseline from donnees_merged
df['comcod'] = df['comcod'].astype(str).str.upper().str.zfill(5)

# --- Map arrondissements to their main communes (5-char codes) ---
arrondissement_mapping = {
    **{str(code).zfill(5): "75056" for code in range(75101, 75121)},  # Paris
    **{str(code).zfill(5): "13055" for code in range(13201, 13217)},  # Marseille
    **{str(code).zfill(5): "69123" for code in range(69381, 69390)},  # Lyon
}
df['comcod'] = df['comcod'].map(lambda x: arrondissement_mapping.get(x, x))

# --- Diagnostics: commune coverage baseline vs health data ---
pop_communes = df['comcod'].dropna().unique()
health_communes = health_data['insee'].dropna().unique()

print(f"Baseline unique communes in donnees_merged (5-char): {len(pop_communes)}")
print(f"Unique communes in health_data (5-char): {len(health_communes)}")

pop_only = set(pop_communes) - set(health_communes)
health_only = set(health_communes) - set(pop_communes)
print(f"Communes present only in population: {len(pop_only)}")
print(f"Communes present only in health: {len(health_only)}")

# --- Prepare commune x age population (aggregate IRIS to commune; no IRIS distribution) ---
comm_age = (
    df.groupby(["comcod", "age"], as_index=False)["pop2019"]
    .sum()
    .rename(columns={"pop2019": "pop_comm_age"})
)

# Merge commune x age population with commune health totals (by disease)
merged = comm_age.merge(
    health_data[['insee', 'disease', 'absolute_incidence', 'inc_100000']],
    left_on="comcod",
    right_on="insee",
    how="left"
)

print(f"Communes retained after merge (based on donnees_merged): {merged['comcod'].nunique()}")

# --- Define disease-specific age ranges (keep validated age stratification) ---
disease_age_groups = {
    "Hypertension": (18, 99),
    "Lung Cancer": (35, 99),
    "Asthma in children": (0, 17),
    "Asthma in adult": (18, 39),
    "COPD": (40, 99),
    "ALRI in children": (0, 12),
    "Stroke": (35, 99),
    "IHD events": (30, 99),
    "Diabetes T2": (45, 99),
}

# --- Allocate commune totals to ages within each commune (no IRIS allocation) ---
merged["absolute_incidence_iris"] = 0.0  # keep column name for downstream compatibility

for disease, (amin, amax) in disease_age_groups.items():
    # mask for disease and valid age range
    mask = (merged["disease"] == disease) & (merged["age"].between(amin, amax))

    # denominator: population per (comcod, disease) across valid ages
    denom = (
        merged.loc[mask]
        .groupby(["comcod", "disease"])["pop_comm_age"]
        .transform("sum")
    )

    share = merged.loc[mask, "pop_comm_age"] / denom.replace(0, np.nan)

    merged.loc[mask, "absolute_incidence_iris"] = (
            share.fillna(0) * pd.to_numeric(merged.loc[mask, "absolute_incidence"], errors="coerce").fillna(0)
    )

# Outside valid age ranges set to zero (already default), ensure numeric
merged["absolute_incidence_iris"] = pd.to_numeric(merged["absolute_incidence_iris"], errors="coerce").fillna(0)

# --- Verification ---
print("\nDisease-specific totals before and after distribution:")
print("-" * 80)
print(f"{'Disease':<20} {'Before':>15} {'After':>15} {'Difference':>15} {'Relative %':>15}")
print("-" * 80)

for disease in disease_age_groups.keys():
    before = health_data.loc[health_data["disease"] == disease, "absolute_incidence"].sum()
    after = merged.loc[merged["disease"] == disease, "absolute_incidence_iris"].sum()
    diff = after - before
    rel = (diff / before * 100) if before > 0 else 0
    print(f"{disease:<20} {before:>15.2f} {after:>15.2f} {diff:>15.2f} {rel:>15.2f}")

total_before = health_data["absolute_incidence"].sum()
total_after = merged["absolute_incidence_iris"].sum()
diff = total_after - total_before
rel = (diff / total_before * 100) if total_before > 0 else 0
print("-" * 80)
print(f"{'TOTAL':<20} {total_before:>15.2f} {total_after:>15.2f} {diff:>15.2f} {rel:>15.2f}")

# Keep only commune-level aggregated structure (comcod, age, disease)
# Note: If fully aggregated at commune (no age), uncomment the following line:
# merged = merged.groupby(['comcod', 'disease'], as_index=False)['absolute_incidence_iris'].sum()


Baseline unique communes in donnees_merged (5-char): 33787
Unique communes in health_data (5-char): 35228
Communes present only in population: 0
Communes present only in health: 1441
Communes retained after merge (based on donnees_merged): 33787

Disease-specific totals before and after distribution:
--------------------------------------------------------------------------------
Disease                       Before           After      Difference      Relative %
--------------------------------------------------------------------------------
Hypertension               712414.50       697848.75       -14565.75           -2.04
Lung Cancer                 39634.80        38760.66         -874.14           -2.21
Asthma in children         201989.25       198422.25        -3567.00           -1.77
Asthma in adult             92322.25        90681.50        -1640.75           -1.78
COPD                       194491.25       190100.75        -4390.50           -2.26
ALRI in children          

In [ ]:
#Distribution without age filtering for national calculation (No age distribution)
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# --- Load health data ---
health_data = pd.read_csv(r"C:\Users\aysharma\Eqis_codes\data\Morbidity data check\final_health_data.csv")
health_data['insee'] = health_data['insee'].astype(str).str.zfill(9)

# --- Preprocess population and mortality data ---
df = donnees_merged.copy()
df['comcod'] = df['comcod'].astype(str).str.zfill(9)

# --- Map arrondissements to their main communes at IRIS level (keep 9-char codes consistently) ---
arrondissement_mapping = {
    **{str(code).zfill(9): "75056".zfill(9) for code in range(75101, 75121)},  # Paris -> 75056
    **{str(code).zfill(9): "13055".zfill(9) for code in range(13201, 13217)},  # Marseille -> 13055
    **{str(code).zfill(9): "69123".zfill(9) for code in range(69381, 69390)},  # Lyon -> 69123
}
df['comcod'] = df['comcod'].map(lambda x: arrondissement_mapping.get(x, x))
df['comcod'] = df['comcod'].astype(str).str.zfill(9)

# --- Allocate commune totals to IRIS (pure spatial proportion, no age-based proportioning) ---
iris_allocations = []
diseases = sorted(health_data["disease"].dropna().unique().tolist())

for disease in diseases:
    # Commune-level totals for this disease
    com_cases = (
        health_data.loc[health_data["disease"] == disease, ["insee", "absolute_incidence"]]
        .rename(columns={"insee": "comcod"})
        .copy()
    )
    com_cases["comcod"] = com_cases["comcod"].astype(str).str.zfill(9)
    if com_cases.empty:
        continue

    # Relevant IRIS population for those communes (all ages pooled)
    df_comm = df[df["comcod"].isin(com_cases["comcod"])].copy()
    if df_comm.empty:
        continue

    iris_pop = (
        df_comm.groupby(["comcod", "iriscod"], as_index=False)["pop2019"]
        .sum()
        .rename(columns={"pop2019": "pop_iris_total"})
    )
    iris_pop["commune_pop"] = iris_pop.groupby("comcod")["pop_iris_total"].transform("sum")
    iris_pop["share"] = iris_pop["pop_iris_total"] / iris_pop["commune_pop"].replace(0, np.nan)

    # Merge with commune-level absolute incidence and distribute by spatial share
    alloc = iris_pop.merge(com_cases, on="comcod", how="left")
    alloc["absolute_incidence_iris"] = alloc["share"].fillna(0) * pd.to_numeric(alloc["absolute_incidence"],
                                                                                errors="coerce").fillna(0)
    alloc["disease"] = disease
    alloc["pop2019"] = alloc["pop_iris_total"]

    iris_allocations.append(alloc[["comcod", "iriscod", "disease", "absolute_incidence_iris", "pop2019"]])

# Final distributed table (one row per IRIS × disease)
iris_allocations = pd.concat(iris_allocations, ignore_index=True) if iris_allocations else pd.DataFrame(
    columns=["comcod", "iriscod", "disease", "absolute_incidence_iris", "pop2019"]
)

# --- Conservation scaling to ensure per-disease totals match (<= 2% diff target) ---
for disease in diseases:
    before = health_data.loc[health_data["disease"] == disease, "absolute_incidence"].sum()
    after = iris_allocations.loc[iris_allocations["disease"] == disease, "absolute_incidence_iris"].sum()
    if pd.notna(before) and before > 0 and pd.notna(after) and after > 0:
        scale = before / after
        if not np.isclose(scale, 1.0, rtol=0.02, atol=0):
            mask_d = iris_allocations["disease"] == disease
            iris_allocations.loc[mask_d, "absolute_incidence_iris"] *= scale

# --- Verification ---
print("\nDisease-specific totals before and after distribution (post-scaling):")
print("-" * 80)
print(f"{'Disease':<20} {'Before':>15} {'After':>15} {'Difference':>15} {'Relative %':>15}")
print("-" * 80)

for disease in diseases:
    before = health_data.loc[health_data["disease"] == disease, "absolute_incidence"].sum()
    after = iris_allocations.loc[iris_allocations["disease"] == disease, "absolute_incidence_iris"].sum()
    diff = after - before
    rel = (diff / before * 100) if before > 0 else 0
    print(f"{disease:<20} {before:>15.2f} {after:>15.2f} {diff:>15.2f} {rel:>15.2f}")

total_before = health_data["absolute_incidence"].sum()
total_after = iris_allocations["absolute_incidence_iris"].sum()
diff = total_after - total_before
rel = (diff / total_before * 100) if total_before > 0 else 0
print("-" * 80)
print(f"{'TOTAL':<20} {total_before:>15.2f} {total_after:>15.2f} {diff:>15.2f} {rel:>15.2f}")

# Assign final result
donnees_merged_new = iris_allocations.copy()


In [ ]:
# --- Define disease-specific age ranges ---
disease_age_groups = {
    "Hypertension": (18, 99),
    "Lung Cancer": (35, 99),
    "Asthma in children": (0, 17),
    "Asthma in adult": (18, 39),
    "COPD": (40, 99),
    "ALRI in children": (0, 12),
    "Stroke": (35, 99),
    "IHD events": (30, 99),
    "Diabetes T2": (45, 99),
}

# Calculate the sum of absolute incidence from donnees_merged for each disease and age group
age_group_sums = {}

for disease, (amin, amax) in disease_age_groups.items():
    filtered_data = merged[
        (merged['disease'] == disease) &
        (merged['age'] >= amin) &
        (merged['age'] <= amax)
        ]
    total_sum = filtered_data['absolute_incidence_iris'].sum()
    age_group_sums[disease] = total_sum

# Display the results
print("Sum of absolute incidence by disease and age group:")
for disease, total_sum in age_group_sums.items():
    print(f"{disease}: {total_sum}")

In [8]:
# Sanity check 2: Group by disease and age to see distribution of cases by age
import matplotlib.pyplot as plt
import os

# Create the directory
output_dir = os.path.join("data","2-output-data", "age-wise incidence")
os.makedirs(output_dir, exist_ok=True)
donnees_merged_new= merged.copy()
cases_distribution = donnees_merged_new.groupby(['disease', 'age'])['absolute_incidence_iris'].sum().reset_index()

# Plot distribution for each disease
diseases = cases_distribution['disease'].unique()

for disease in diseases:
    disease_data = cases_distribution[cases_distribution['disease'] == disease]
    plt.figure(figsize=(10, 6))
    plt.bar(disease_data['age'], disease_data['absolute_incidence_iris'], color='skyblue')
    plt.title(f'Age-wise Distribution of Cases in 2019 for {disease}')
    plt.xlabel('Age')
    plt.ylabel('Number of Cases')
    plt.tight_layout()
    plot_path = os.path.join(output_dir, f"{disease}_age_distribution.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()


In [ ]:
#Sensitivity Analysis for Sanity check based on National Morbidity and No age (SPF method)
import warnings
import logging
import os
import pandas as pd
import numpy as np

warnings.simplefilter(action='ignore', category=FutureWarning)
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

SPF_THRESHOLDS = {"ug_PM25_RH50": 3, "ug_NO2": 1}

scenarios = ["s1"]
pollutants = ["ug_PM25_RH50"]
annees = [2019]


def get_disease_list(pol):
    if pol == "ug_NO2":
        return ["Asthma in adult", "Asthma in children", "ALRI in children"]
    elif pol == "ug_PM25_RH50":
        return ["Lung Cancer", "Asthma in children", "COPD",
                "Stroke", "IHD events", "Hypertension", "Diabetes T2"]
    else:
        return []


def process_combination(params, donnees_exportees_transformed, grille_combinee, donnees_merged_new):
    sc, annee, pol = params
    logging.info(f"Processing combination: Scenario={sc}, Pollutant={pol}, Year={annee}")
    results = []

    try:
        # --- Exposure calculation ---
        conc_points = coordo_sherpa(sc, pol, annee)
        #conc_ineris = coordo_ineris_new(pol)
        conc_ineris = coordo_ineris_avg(pol)
        conc_corrigee = correction(conc_points, conc_ineris)
        donnees_expo = expo(donnees_exportees_transformed, conc_corrigee, grille_combinee)
        if "meanconc" not in donnees_expo.columns:
            logging.error("Exposure table missing 'meanconc' column.")
            return pd.DataFrame()
        logging.info(
            f"Mean concentration for {pol} in {annee}: {pd.to_numeric(donnees_expo['meanconc'], errors='coerce').mean()}")

        # --- National morbidity avoided ---
        relevant_diseases = get_disease_list(pol)
        relevant_configs = [cfg for cfg in morb_config if cfg["disease"] in relevant_diseases]

        for cfg in relevant_configs:
            disease = cfg["disease"]
            df = donnees_merged_new.copy()
            df["disease_norm"] = df["disease"].apply(normalize_string)
            disease_norm_target = normalize_string(disease)
            df = df[df["disease_norm"] == disease_norm_target].copy()

            if df.empty:
                logging.warning(f"No rows matched disease filter for {disease}")
                continue

            # Assign RR
            if pol == "ug_PM25_RH50" and cfg.get("rr_ug_PM25_RH50") is not None:
                rr, rr_low, rr_high = cfg["rr_ug_PM25_RH50"]
                pol_cfg = "ug_PM25_RH50"
            elif pol == "ug_NO2" and cfg.get("rr_ug_NO2") is not None:
                rr, rr_low, rr_high = cfg["rr_ug_NO2"]
                pol_cfg = "ug_NO2"
            else:
                logging.warning(f"No RR available for disease {disease} and pollutant {pol}")
                continue

            threshold = SPF_THRESHOLDS[pol_cfg]

            # Merge with exposure
            df_expo = donnees_expo.drop(columns=["geometry"], errors="ignore")
            needed_cols = {"iriscod", "meanconc"}
            if not needed_cols.issubset(df_expo.columns) or "iriscod" not in df.columns:
                logging.error("Missing required columns for merge ('iriscod'/'meanconc').")
                continue
            df_expo = df_expo[["iriscod", "meanconc"]].copy()

            df_merged = pd.merge(df, df_expo, on="iriscod", how="left")
            if "pop2019" not in df_merged.columns:
                logging.error(f"'pop2019' column missing for disease {disease}")
                continue

            # Coerce numeric and handle missing
            df_merged["pop2019"] = pd.to_numeric(df_merged["pop2019"], errors="coerce").fillna(0)
            df_merged["meanconc"] = pd.to_numeric(df_merged["meanconc"], errors="coerce").fillna(0)

            total_pop = df_merged["pop2019"].sum()
            if total_pop == 0:
                logging.warning(f"Total population is zero for disease {disease}")
                continue

            # Use exceedance above SPF threshold (do NOT rely on meandelta which can be baseline-difference = 0)
            df_merged["excess"] = (df_merged["meanconc"] - threshold).clip(lower=0)

            weighted_delta = (df_merged["excess"] * df_merged["pop2019"]).sum() / total_pop
            delta = float(max(0.0, weighted_delta))

            beta_morb = np.log(rr)
            AF = 1 - np.exp(-beta_morb * delta)

            baseline_cases = pd.to_numeric(df["absolute_incidence_iris"], errors="coerce").fillna(0).sum()
            if baseline_cases == 0:
                logging.warning(f"Baseline cases are zero for disease {disease}")
            cases_avoided = AF * baseline_cases
            yld_avoided = cases_avoided * float(cfg.get("disability_weight", 0)) * float(cfg.get("duration", 0))

            results.append([disease, cases_avoided, yld_avoided, delta, pol_cfg, rr])

        # Wrap results into dataframe
        national_df = pd.DataFrame(
            results,
            columns=["disease", "cases_avoided", "yld_avoided", "delta", "pollutant", "RR"]
        )

        if not national_df.empty:
            total_row = pd.DataFrame([[
                "TOTAL",
                national_df["cases_avoided"].sum(),
                national_df["yld_avoided"].sum(),
                national_df["delta"].mean(),
                pol,
                np.nan
            ]], columns=national_df.columns)
            national_df = pd.concat([national_df, total_row], ignore_index=True)

        logging.info(f"Finished processing for Scenario={sc}, Pollutant={pol}, Year={annee}")
        return national_df

    except Exception as e:
        logging.error(f"Error in processing for Scenario={sc}, Pollutant={pol}, Year={annee}: {e}")
        return pd.DataFrame()


# --- Main execution ---
if __name__ == "__main__":
    params_list = [(sc, annee, pol) for sc in scenarios for annee in annees for pol in pollutants]
    output_dir = os.path.join("data", "2-output-data")
    os.makedirs(output_dir, exist_ok=True)
    national_output = os.path.join(output_dir, "national_summary.xlsx")

    with pd.ExcelWriter(national_output, engine="openpyxl") as writer:
        wrote_any = False
        for params in params_list:
            logging.info(f"Starting processing for params: {params}")
            national_df = process_combination(params, donnees_exportees_transformed, grille_combinee,
                                              donnees_merged_new)
            if not national_df.empty:
                sheet_name = "PM25" if params[2] == "ug_PM25_RH50" else "NO2"
                national_df.to_excel(writer, sheet_name=sheet_name, index=False)
                logging.info(f"Saved sheet '{sheet_name}'")
                wrote_any = True

        if not wrote_any:
            pd.DataFrame({"message": ["No results generated"]}).to_excel(writer, sheet_name="Empty", index=False)

    logging.info("Successfully processed all combinations.")


In [9]:
#Code to conduct morbidity estimation using SPF method at COMCOD level with SpF thresholds
# Assumes morb_config and helper functions are imported from helper file
SPF_THRESHOLDS = {"ug_PM25_RH50": 5, "ug_NO2": 10}
LIFE_EXPECTANCY = {2019: 80, 2030: 84, 2050: 86}

#Code to conduct morbidity estimation using SPF method at COMCOD level with SpF thresholds
def morbidity_spf_method(donnees_merged_new, commune_conc, annee, pol, disease, morb_config):
    """
    SPF-style morbidity estimation including mortality for all ages using predefined RR values.
    Works at commune (comcod) level for exposure merge and aggregation.
    """
    import numpy as np
    import pandas as pd
    import logging

    logger = logging.getLogger(__name__)

    # --- Relative risks for mortality ---
    RR_PM25 = 1.15
    RR_PM25_high = 1.25
    RR_PM25_low = 1.05
    RR_NO2 = 1.023
    RR_NO2_high = 1.037
    RR_NO2_low = 1.008

    RR_mort_map = {
        "ug_PM25_RH50": RR_PM25,
        "ug_PM25_RH50_high": RR_PM25_high,
        "ug_PM25_RH50_low": RR_PM25_low,
        "ug_NO2": RR_NO2,
        "ug_NO2_high": RR_NO2_high,
        "ug_NO2_low": RR_NO2_low
    }

    try:
        annee = int(annee)
        df_pop = donnees_merged_new.copy()
        df_expo = commune_conc.copy()  #INERIS conc directly no correction
        #Drop geometry column from df_expo if exist
        df_expo.drop(columns=["geometry"], errors="ignore", inplace=True)

        # Ensure comcod exists and is comparable (5 or 9 length possible; keep as string)
        df_pop["comcod"] = df_pop["comcod"].astype(str).str.upper()
        if "comcod" not in df_expo.columns:
            return pd.DataFrame({"error": ["Missing required column 'comcod' in exposure data"]})
        df_expo["comcod"] = df_expo["comcod"].astype(str).str.upper()

        # Keep only necessary exposure columns (meanconc required)
        keep_cols = [c for c in ["comcod", "meanconc"] if c in df_expo.columns]
        if "meanconc" not in keep_cols:
            return pd.DataFrame({"error": ["Exposure data must contain 'meanconc' at comcod level"]})
        df_expo = df_expo[keep_cols].copy()

        # Force meanconc numeric BEFORE merge (prevents object dtype propagation)
        df_expo["meanconc"] = pd.to_numeric(df_expo["meanconc"], errors="coerce").astype("float64")

        # Merge at comcod level
        merged_data = pd.merge(df_pop, df_expo, on="comcod", how="left")

        # Normalize disease
        merged_data["disease_norm"] = merged_data["disease"].apply(normalize_string)
        disease_norm_target = normalize_string(disease)
        merged_data = merged_data[merged_data["disease_norm"] == disease_norm_target].copy()
        if merged_data.empty:
            logger.warning(f"No rows for disease {disease}")
            return pd.DataFrame()

        # --- Morbidity config ---
        cfg, rr_key = find_matching_morbidity_config(disease, pol, morb_config)
        if cfg is None or rr_key is None or not cfg.get(rr_key):
            logger.warning(f"Missing morbidity config for {disease}, {pol}")
            return pd.DataFrame()

        RR_morb, RR_morb_lo, RR_morb_hi = cfg[rr_key]

        # Population columns
        #pop_col = f"pop{annee}" if f"pop{annee}" in merged_data.columns else "pop2019"
        #mort_col = f"mort{annee}" if f"mort{annee}" in merged_data.columns else "mort2019"
        #Stable population columns
        pop_col = "pop2019"
        mort_col = "mort2019"

        # Safe numeric conversions (avoid scalar .get defaults)
        if "age" in merged_data.columns:
            merged_data["age"] = pd.to_numeric(merged_data["age"], errors="coerce")
        else:
            merged_data["age"] = np.nan

        if pop_col in merged_data.columns:
            merged_data[pop_col] = pd.to_numeric(merged_data[pop_col], errors="coerce").fillna(0).astype("float64")
        else:
            merged_data[pop_col] = 0.0

        if mort_col in merged_data.columns:
            merged_data[mort_col] = pd.to_numeric(merged_data[mort_col], errors="coerce").fillna(0).astype("float64")
        else:
            merged_data[mort_col] = 0.0

        if "absolute_incidence_iris" in merged_data.columns:
            merged_data["absolute_incidence_iris"] = pd.to_numeric(
                merged_data["absolute_incidence_iris"], errors="coerce"
            ).fillna(0).astype("float64")
        else:
            merged_data["absolute_incidence_iris"] = 0.0

        if "meanconc" in merged_data.columns:
            merged_data["meanconc"] = pd.to_numeric(merged_data["meanconc"], errors="coerce").fillna(0).astype(
                "float64")
        else:
            merged_data["meanconc"] = 0.0

        # ΔC above threshold (use meanconc - threshold; no meandelta here)
        pol_base = get_pollutant_base(pol)
        spf_threshold = float(SPF_THRESHOLDS.get(pol_base, 0.0))

        # Avoid pandas/numexpr object-dtype arithmetic by computing via NumPy float64 arrays
        meanconc_arr = merged_data["meanconc"].to_numpy(dtype="float64", copy=False)
        merged_data["meandelta_pos"] = np.maximum(meanconc_arr - spf_threshold, 0.0)
        #Sensitivity Check with all values to 5
        merged_data["meandelta_pos"] = np.full_like(meanconc_arr, 5.0, dtype="float64")

        # --- Morbidity AF ---
        beta_c, beta_lo, beta_hi = np.log(RR_morb), np.log(RR_morb_lo), np.log(RR_morb_hi)
        merged_data["AF_central"] = 1 - np.exp(-beta_c * merged_data["meandelta_pos"] / 10)
        merged_data["AF_low"] = 1 - np.exp(-beta_lo * merged_data["meandelta_pos"] / 10)
        merged_data["AF_high"] = 1 - np.exp(-beta_hi * merged_data["meandelta_pos"] / 10)

        # Baseline cases (convert rate to counts if needed)
        K = float(cfg.get("incidence_per", 1.0))
        if K == 1:
            merged_data["baseline_cases"] = merged_data["absolute_incidence_iris"]
        else:
            merged_data["baseline_cases"] = merged_data["absolute_incidence_iris"] * merged_data[pop_col] / K

        #Calculate Prevalence for YLD and DALY
        merged_data["Prevelance"] = merged_data["baseline_cases"] / merged_data[pop_col].replace(0, np.nan)
        merged_data["Prevelance"] = merged_data["Prevelance"].fillna(0.0)

        # Avoided morbidity
        merged_data["avoided_cases_central"] = merged_data["baseline_cases"] * merged_data["AF_central"]
        merged_data["avoided_cases_low"] = merged_data["baseline_cases"] * merged_data["AF_low"]
        merged_data["avoided_cases_high"] = merged_data["baseline_cases"] * merged_data["AF_high"]

        # --- Age aggregation (still report by age; exposure merged at comcod) ---
        min_age = int(cfg.get("age_min", 0))
        max_age = int(cfg.get("age_max", 99))
        df_age = merged_data[(merged_data["age"] >= min_age) & (merged_data["age"] <= max_age)].copy()

        grouped = df_age.groupby("age", as_index=False).agg(
            {
                pop_col: "sum",
                "baseline_cases": "sum",
                "avoided_cases_central": "sum",
                "avoided_cases_low": "sum",
                "avoided_cases_high": "sum",
            }
        )
        ages = np.arange(min_age, max_age + 1)
        donnees = pd.DataFrame({"age": ages}).merge(grouped, on="age", how="left").fillna(0)

        # Totals
        baseline_total_cases = df_age["baseline_cases"].sum()
        tot_avoided_cases = df_age["avoided_cases_central"].sum()
        donnees["baseline_morbidity"] = baseline_total_cases
        donnees["tot_avoided_cases"] = tot_avoided_cases
        donnees["percent_avoided_morbidity"] = (
            tot_avoided_cases / baseline_total_cases * 100 if baseline_total_cases > 0 else 0
        )

        # --- Mortality AF using RR_mort_map ---
        RR_mort_central = RR_mort_map.get(pol, 1.0)
        RR_mort_low = RR_mort_map.get(f"{pol}_low", RR_mort_central)
        RR_mort_high = RR_mort_map.get(f"{pol}_high", RR_mort_central)

        b_c, b_lo, b_hi = np.log(RR_mort_central), np.log(RR_mort_low), np.log(RR_mort_high)

        merged_data[f"mort_central_{annee}"] = merged_data[mort_col] * (
                1 - np.exp(-b_c * merged_data["meandelta_pos"] / 10)
        )
        merged_data[f"mort_low_{annee}"] = merged_data[mort_col] * (
                1 - np.exp(-b_lo * merged_data["meandelta_pos"] / 10)
        )
        merged_data[f"mort_high_{annee}"] = merged_data[mort_col] * (
                1 - np.exp(-b_hi * merged_data["meandelta_pos"] / 10)
        )

        # Aggregate mortality by age
        mort_grouped = (
            merged_data.groupby("age", as_index=False)[
                [f"mort_central_{annee}", f"mort_low_{annee}", f"mort_high_{annee}"]
            ]
            .sum()
            .rename(
                columns={
                    f"mort_central_{annee}": "mortality_avoided_central",
                    f"mort_low_{annee}": "mortality_avoided_low",
                    f"mort_high_{annee}": "mortality_avoided_high",
                }
            )
        )
        donnees = donnees.merge(mort_grouped, on="age", how="left").fillna(0)

        # --- YLL ---
        esp_dict = {2019: 80, 2030: 84, 2050: 86}
        esp = float(esp_dict.get(annee, 80))
        merged_data["remaining_life_expectancy"] = (esp - merged_data["age"]).clip(lower=0)
        for col in ["central", "low", "high"]:
            merged_data[f"YLL_{col}"] = merged_data[f"mort_{col}_{annee}"] * merged_data["remaining_life_expectancy"]

        YLL_grouped = merged_data.groupby("age", as_index=False)[["YLL_central", "YLL_low", "YLL_high"]].sum()
        donnees = donnees.merge(YLL_grouped, on="age", how="left").fillna(0)

        # --- YLD & DALY ---
        def _as_three(x, default=1.0):
            if x is None:
                return (float(default), float(default), float(default))
            if isinstance(x, (list, tuple, np.ndarray, pd.Series)):
                if len(x) >= 3:
                    return (float(x[0]), float(x[1]), float(x[2]))
                if len(x) == 1:
                    v = float(x[0])
                    return (v, v, v)
                if len(x) == 2:
                    v0, v1 = float(x[0]), float(x[1])
                    return (v0, v0, v1)
            v = float(x)
            return (v, v, v)

        dw_c, dw_l, dw_h = _as_three(cfg.get("disability_weight", 1.0), default=1.0)
        dur_c, dur_l, dur_h = _as_three(cfg.get("duration", 1.0), default=1.0)

        donnees["YLD_central"] = donnees["avoided_cases_central"] * dw_c * dur_c
        donnees["YLD_low"] = donnees["avoided_cases_low"] * dw_l * dur_l
        donnees["YLD_high"] = donnees["avoided_cases_high"] * dw_h * dur_h

        donnees["DALY_central"] = donnees["YLL_central"] + donnees["YLD_central"]
        donnees["DALY_low"] = donnees["YLL_low"] + donnees["YLD_low"]
        donnees["DALY_high"] = donnees["YLL_high"] + donnees["YLD_high"]

        # Metadata
        donnees["disease"] = disease
        donnees["pollutant"] = pol
        donnees["year"] = annee
        donnees["RR_morbidity"] = RR_morb
        donnees["spf_threshold"] = spf_threshold

        return donnees.sort_values("age").reset_index(drop=True)

    except Exception as e:
        logger.exception(f"Error in morbidity_spf_method: {e}")
        return pd.DataFrame({"error": [str(e)]})

In [10]:
#Main analysis to compare with SPF cases avoided with WHO thresholds
import warnings
import logging
import os

warnings.simplefilter(action='ignore', category=FutureWarning)
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Define scenarios, pollutants, and years
scenarios = ["s2"]
pollutants = ["ug_NO2"]  # Use ug_NO2 only for Respiratory Analysis, else "ug_PM25_RH50"
annees = [2019]  # Changed string to integer


def get_disease_list(pol):
    """Return appropriate diseases for the pollutant."""
    if pol == "ug_NO2":
        return ['Asthma in adult', 'Asthma in children', 'ALRI in children']
    elif pol == "ug_PM25_RH50":
        return ['Lung Cancer', 'Asthma in children', 'COPD', 'Stroke', 'IHD events', 'Hypertension', 'Diabetes T2']
    else:
        return []

def process_combination(params, donnees_exportees_transformed, grille_combinee, merged):
    sc, annee, pol = params
    logging.info(f"Processing combination: Scenario={sc}, Pollutant={pol}, Year={annee}")
    try:
        # These functions must be defined elsewhere and imported
        #conc_points = coordo_sherpa(sc, pol, annee)
        conc_ineris = coordo_ineris_avg(pol) #with 4 year avg values of INERIS cartotique
        commune_conc = merge_conc_with_communes(conc_ineris)
        #coordo_ineris_new(pol) #forNO2 and coordo_ineris_PM(pol) for PM2.5
        #conc_corrigee = correction(conc_points, conc_ineris)
        #donnees_expo = expo(donnees_exportees_transformed, conc_corrigee, grille_combinee)
        logging.info(f"Mean concentration for {pol} in {annee}: {commune_conc['meanconc'].mean()}")
        output_dir = os.path.join(r"C:\Users\aysharma\Eqis_codes\data\2-output-data\disease_specific", pol, str(annee))
        os.makedirs(output_dir, exist_ok=True)
        output_excel_path = os.path.join(output_dir, "all_diseases_summary_S1.xlsx")

        diseases = get_disease_list(pol)
        # Save each disease result as a separate sheet in the same Excel file
        with pd.ExcelWriter(output_excel_path, engine='openpyxl', mode='w') as writer:
            for disease in diseases:
                try:
                    logging.info(f"Calculating base (non-MC) morbidity for {disease}, Year={annee}")
                    result_df = morbidity_spf_method(
                        merged,
                        commune_conc,
                        annee,
                        pol,
                        disease,
                        morb_config  # Pass config!
                    )
                    if result_df is not None and not result_df.empty:
                        sheet_name = disease.replace(' ', '_')[:31]
                        result_df.to_excel(writer, sheet_name=sheet_name, index=False)
                except Exception as e:
                    logging.error(f"Error processing disease {disease}: {str(e)}")

        logging.info(f"All disease results saved to {output_excel_path}")
        logging.info(f"Finished processing for Scenario={sc}, Pollutant={pol}, Year={annee}")

    except Exception as e:
        logging.error(f"Error in main processing for Scenario={sc}, Pollutant={pol}, Year={annee}: {e}")


# Main execution
if __name__ == "__main__":
    # Ensure these variables are defined and loaded properly
    # donnees_exportees_transformed = ...
    # grille_combinee = ...
    # donnees_merged_new = ...
    params_list = [(sc, annee, pol) for sc in scenarios for annee in annees for pol in pollutants]

    for params in params_list:
        logging.info(f"Starting processing for params: {params}")
        process_combination(params, donnees_exportees_transformed, grille_combinee, merged)

    logging.info("Successfully processed all combinations.")


INFO:root:Starting processing for params: ('s2', 2019, 'ug_NO2')
INFO:root:Processing combination: Scenario=s2, Pollutant=ug_NO2, Year=2019


Starting coordo_ineris_avg function
Year 2016: valid grid cells = 533248, mean = 8.304, min = 0.098, max = 38.236
Year 2016: interpolated valid cells = 1644201
Year 2017: valid grid cells = 533248, mean = 7.917, min = 0.068, max = 37.900
Year 2017: interpolated valid cells = 1644201
Year 2018: valid grid cells = 533248, mean = 7.442, min = 0.111, max = 36.115
Year 2018: interpolated valid cells = 1644201
Year 2019: valid grid cells = 533248, mean = 7.079, min = 0.091, max = 34.021
Year 2019: interpolated valid cells = 1644201
Final meanconc: min = 0.150, max = 36.292, mean = 8.839, valid cells = 1644201
Finished coordo_ineris_avg function


INFO:root:Mean concentration for ug_NO2 in 2019: 8.506360299723385
INFO:root:Calculating base (non-MC) morbidity for Asthma in adult, Year=2019


Simple mean of commune means: 8.5064
Loaded defined RR values for mortality


INFO:root:Calculating base (non-MC) morbidity for Asthma in children, Year=2019


Loaded defined RR values for mortality


INFO:root:Calculating base (non-MC) morbidity for ALRI in children, Year=2019


Loaded defined RR values for mortality


INFO:root:All disease results saved to C:\Users\aysharma\Eqis_codes\data\2-output-data\disease_specific\ug_NO2\2019\all_diseases_summary_S1.xlsx
INFO:root:Finished processing for Scenario=s2, Pollutant=ug_NO2, Year=2019
INFO:root:Successfully processed all combinations.
